# Phase 4: PIXC visualization and manual transect inspection

This notebook demonstrates local plan-view visualization and the user-supplied transect API. It never searches for or downloads SWOT data. The `channel_extent_candidate` profile is **experimental / unvalidated**. No bank, branch, island, or channel width is inferred or calculated.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import swot_pixc_lab as spl

## Open verified local PIXC files

Paths are repository-relative. Replace them with your own verified local PIXC files when adapting the workflow. Add one cycle/pass file group per observation to inspect the same AOI across dates. Each group must come from a reviewed discovery manifest and include every intended AOI-intersecting tile; the arbitrary display label is not provenance. The guard below skips missing groups, prints the opened NetCDF source table for review, and cannot initiate discovery or download.

In [ ]:
KOSHI_AOI = (86.87, 26.49, 87.20, 26.90)
KOSHI_EXAMPLE_LABEL = "2024-01-14 · cycle 009 / pass 286"
LOCAL_OBSERVATION_GROUPS = {
    KOSHI_EXAMPLE_LABEL: (
        Path("data/koshi_phase1")
        / "SWOT_L2_HR_PIXC_009_286_107L_20240114T081432_20240114T081443_PGD0_01.nc",
        Path("data/koshi_phase1")
        / "SWOT_L2_HR_PIXC_009_286_108L_20240114T081442_20240114T081453_PGD0_01.nc",
    ),
    # Add another label: (tile_path, ...) entry for each additional observation.
}

available_observation_groups = {}
for label, paths in LOCAL_OBSERVATION_GROUPS.items():
    missing = [path for path in paths if not path.is_file()]
    if missing:
        print(f"Skipping {label}; no download will be attempted.")
        for path in missing:
            print(f"  missing: {path}")
    else:
        available_observation_groups[label] = paths

In [ ]:
reviews = {}
for label, paths in available_observation_groups.items():
    current_observation = spl.open_pixc(
        paths,
        aoi=KOSHI_AOI,
        variables=spl.PHASE3_POINT_VARIABLES,
    )
    current_candidate = spl.apply_qc(
        current_observation, profile="channel_extent_candidate"
    )
    reviews[label] = (current_observation, current_candidate)
    print(f"\nUser display label: {label}")
    print(
        current_observation.source_table[
            ["filename", "cycle", "pass", "tile", "points_after"]
        ].to_string(index=False)
    )
    if label == KOSHI_EXAMPLE_LABEL:
        assert current_observation.pixel_count == 2_230_522
        water_count = sum(
            current_observation.classification_counts.get(value, 0)
            for value in range(3, 8)
        )
        assert water_count == 280_201
        assert current_candidate.filtered.sizes["points"] == 280_198
    print(label, current_candidate.profile.label, "—", current_candidate.profile.status)

observation, candidate = reviews.get(KOSHI_EXAMPLE_LABEL, (None, None))

## Compare raw context, documented water classes, and the candidate

For every available observation group, the three panels show raw classes 1–7, documented PIXC water classes 3–7, and the experimental Phase-3 candidate. Classes 3–7 are not asserted to be a validated river boundary. Class 2 remains visible as contextual `land_near_water` in the raw panel. Shared AOI longitude/latitude axes let a scientist compare dates and later supply transect endpoints manually.

In [ ]:
comparison_figures = {}
for label, (current_observation, current_candidate) in reviews.items():
    comparison_figures[label] = spl.plot_classification_comparison(
        current_observation,
        candidate=current_candidate,
        title=f"Koshi PIXC classification review — {label}",
        extent=KOSHI_AOI,
    )

## Inspect another point variable

`plot_pixc_map` also accepts a raw observation, a QC result, or a filtered xarray dataset. This example colors the experimental candidate by water fraction without constructing GeoPandas point geometries.

In [ ]:
if candidate is not None:
    water_fraction_axes = spl.plot_pixc_map(
        candidate,
        color_by="water_frac",
        title="Koshi water-fraction samples — experimental candidate",
    )

## Synthetic user-supplied transect

The following coordinates and pixels are entirely synthetic and are not a Koshi scientific transect. The endpoints are supplied explicitly by the user. The 50 m corridor is a sampling choice—not a river-width measurement.

In [ ]:
SYNTHETIC_ENDPOINTS = ((-93.0020, 45.0000), (-92.9980, 45.0000))
synthetic_pixels = xr.Dataset(
    data_vars={
        "longitude": (
            "points",
            np.array([-93.0018, -93.0010, -93.0000, -92.9990, -92.9982, -93.0000]),
        ),
        "latitude": (
            "points",
            np.array([45.0000, 45.0001, 44.9998, 45.0002, 45.0000, 45.0020]),
        ),
        "classification": ("points", np.array([2, 3, 4, 5, 7, 4], dtype=np.uint8)),
        "height": (
            "points",
            np.array([91.0, 90.8, 90.7, 90.9, 91.1, 92.0], dtype=np.float32),
        ),
        "height_egm2008": (
            "points",
            np.array([45.0, 44.8, 44.7, 44.9, 45.1, 46.0], dtype=np.float32),
        ),
        "water_frac": (
            "points",
            np.array([0.1, 0.5, 1.0, 0.7, 0.4, 1.0], dtype=np.float32),
        ),
        "source_index": ("points", np.array([0, 0, 0, 1, 1, 1], dtype=np.int32)),
        "source_point_index": ("points", np.arange(6, dtype=np.int64)),
    }
)

In [ ]:
synthetic_sample = spl.sample_transect(
    synthetic_pixels,
    transect=SYNTHETIC_ENDPOINTS,
    corridor_half_width_m=50.0,
)
print("Selected synthetic pixels:", synthetic_sample.selected_pixel_count)
print("Classification counts:", synthetic_sample.classification_counts)
print("Corridor half-width (m):", synthetic_sample.corridor_half_width_m)

In [ ]:
diagnostic_figure, diagnostic_axes = plt.subplots(1, 3, figsize=(16, 4.5))
spl.plot_transect_corridor(synthetic_sample, ax=diagnostic_axes[0])
spl.plot_transect_classification(synthetic_sample, ax=diagnostic_axes[1])
spl.plot_transect_height(synthetic_sample, ax=diagnostic_axes[2])
diagnostic_figure.suptitle("Synthetic manual-transect pixel diagnostics")
diagnostic_figure.tight_layout()

## Interpretation limits and next owner input

- `station_m` and `distance_to_transect_m` are local projected distances to the finite, user-supplied line. They do not identify banks or branches.
- PIXC samples are irregular. The diagnostic plots do not interpolate across islands, gaps, or missing observations.
- Corridor membership only identifies pixels near the line; the corridor width is not a channel width.
- The experimental candidate requires visual and scientific validation against independent information.
- No real Koshi transect is selected here. After reviewing the map, the project owner can provide one as `(lon1, lat1) -> (lon2, lat2)`.
- Phase 4 does not perform automatic segmentation, bank detection, width estimation, WSE correction, slope, discharge, or multi-date analysis.